# 📓 Notebook 5 — Feature Engineering
### AirSense AI — Intelligent Urban Air Quality Forecasting & Decision Support System

**Purpose of this notebook**

Turn the cleaned AirSense Feature Cube into a model-ready feature table.
Everything built here is grounded in what Notebook 4 (Statistical Analysis)
actually found — not arbitrary guesses:

- **PM2.5 lag features** at t-1, t-3, t-6, t-12, t-24 (standard autoregressive
  lags, capturing the target's own recent history)
- **Weather lag features** at the *specific* optimal lags found via Granger
  causality in Notebook 4: Wind Speed(t-3), Relative Humidity(t-2),
  Temperature(t-6), Cloud Cover(t-2), Precipitation(t-1)
- **24-hour rolling mean and std** for PM2.5
- **Calendar features**, encoding the categorical ones (Weekday, Season) as
  dummy variables so models can use them numerically
- **MinMaxScaler normalization**, saved for reuse in the dashboard

Wind Direction and raw Precipitation were flagged "Review/Exclude" in
Notebook 4 due to weak correlation — we include Precipitation's lag anyway
since it *did* pass Granger causality, but exclude raw Wind Direction as a
feature (kept available in the cube if you want to revisit it later).

---


## 1. Imports & setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
import joblib

pd.set_option('display.max_columns', None)
print("pandas:", pd.__version__)

pandas: 2.3.3


## 2. Load the AirSense Feature Cube

Same explicit datetime parsing pattern used since Notebook 3 — required
because of mixed UTC offsets from DST transitions.

In [2]:
PROJECT_ROOT = Path(r"C:\Users\pc\Desktop\AirSenseAI")
CUBE_PATH = PROJECT_ROOT / "data" / "processed" / "airsense_cube.csv"
MODELS_DIR = PROJECT_ROOT / "models"

print("Cube file:", CUBE_PATH, "| exists:", CUBE_PATH.exists())

Cube file: C:\Users\pc\Desktop\AirSenseAI\data\processed\airsense_cube.csv | exists: True


In [3]:
df = pd.read_csv(CUBE_PATH)
df['time'] = pd.to_datetime(df['time'], utc=True).dt.tz_convert('America/New_York')
df = df.sort_values('time').reset_index(drop=True)

print("Shape:", df.shape)
df.head()

Shape: (17568, 17)


,time,PM10,PM2.5,O3,NO2,Temperature,Relative Humidity,Wind Speed,Wind Direction,Precipitation,Cloud Cover,Hour,Weekday,Month,Weekend,ISO Week,Season
0,2023-12-31 19:00:00-05:00,27.8,19.3,0,56.8,3.8,65,5.7,235,0.0,65,19,Sunday,12,True,52,Winter
1,2023-12-31 20:00:00-05:00,31.3,21.7,0,55.6,3.8,60,6.0,245,0.0,99,20,Sunday,12,True,52,Winter
2,2023-12-31 21:00:00-05:00,35.5,24.6,0,54.0,3.1,65,6.8,238,0.0,96,21,Sunday,12,True,52,Winter
3,2023-12-31 22:00:00-05:00,40.0,27.8,0,52.5,2.6,68,8.4,245,0.0,73,22,Sunday,12,True,52,Winter
4,2023-12-31 23:00:00-05:00,43.9,30.5,0,51.5,1.9,74,6.3,246,0.0,100,23,Sunday,12,True,52,Winter


## 3. PM2.5 lag features

Standard autoregressive lags: the target's own value 1, 3, 6, 12, and 24
hours ago. These are almost always the strongest predictors in pollutant
forecasting, since PM2.5 tends to persist over several hours.

In [4]:
pm25_lags = [1, 3, 6, 12, 24]

for lag in pm25_lags:
    df[f'PM2.5(t-{lag})'] = df['PM2.5'].shift(lag)

df[['time', 'PM2.5'] + [f'PM2.5(t-{lag})' for lag in pm25_lags]].head(26)

,time,PM2.5,PM2.5(t-1),PM2.5(t-3),PM2.5(t-6),PM2.5(t-12),PM2.5(t-24)
0,2023-12-31 19:00:00-05:00,19.3,NaN,NaN,NaN,NaN,NaN
1,2023-12-31 20:00:00-05:00,21.7,19.3,NaN,NaN,NaN,NaN
2,2023-12-31 21:00:00-05:00,24.6,21.7,NaN,NaN,NaN,NaN
3,2023-12-31 22:00:00-05:00,27.8,24.6,19.3,NaN,NaN,NaN
4,2023-12-31 23:00:00-05:00,30.5,27.8,21.7,NaN,NaN,NaN
5,2024-01-01 00:00:00-05:00,31.9,30.5,24.6,NaN,NaN,NaN
6,2024-01-01 01:00:00-05:00,28.3,31.9,27.8,19.3,NaN,NaN
7,2024-01-01 02:00:00-05:00,22.8,28.3,30.5,21.7,NaN,NaN
8,2024-01-01 03:00:00-05:00,20.4,22.8,31.9,24.6,NaN,NaN
9,2024-01-01 04:00:00-05:00,19.4,20.4,28.3,27.8,NaN,NaN


## 4. Weather lag features — using Notebook 4's optimal lags

Rather than picking arbitrary lag values, we use the exact lags where each
variable's Granger causality was strongest (see Notebook 4, Section 8):

| Variable | Optimal Lag (hours) |
|---|---|
| Wind Speed | 3 |
| Relative Humidity | 2 |
| Temperature | 6 |
| Cloud Cover | 2 |
| Precipitation | 1 |

In [5]:
weather_lag_map = {
    'Wind Speed': 3,
    'Relative Humidity': 2,
    'Temperature': 6,
    'Cloud Cover': 2,
    'Precipitation': 1,
}

for var, lag in weather_lag_map.items():
    df[f'{var}(t-{lag})'] = df[var].shift(lag)

lag_cols = [f'{var}(t-{lag})' for var, lag in weather_lag_map.items()]
df[['time'] + list(weather_lag_map.keys()) + lag_cols].head(10)

,time,Wind Speed,Relative Humidity,Temperature,Cloud Cover,Precipitation,Wind Speed(t-3),Relative Humidity(t-2),Temperature(t-6),Cloud Cover(t-2),Precipitation(t-1)
0,2023-12-31 19:00:00-05:00,5.7,65,3.8,65,0.0,NaN,NaN,NaN,NaN,NaN
1,2023-12-31 20:00:00-05:00,6.0,60,3.8,99,0.0,NaN,NaN,NaN,NaN,0.0
2,2023-12-31 21:00:00-05:00,6.8,65,3.1,96,0.0,NaN,65.0,NaN,65.0,0.0
3,2023-12-31 22:00:00-05:00,8.4,68,2.6,73,0.0,5.7,60.0,NaN,99.0,0.0
4,2023-12-31 23:00:00-05:00,6.3,74,1.9,100,0.0,6.0,65.0,NaN,96.0,0.0
5,2024-01-01 00:00:00-05:00,8.9,76,1.8,100,0.0,6.8,68.0,NaN,73.0,0.0
6,2024-01-01 01:00:00-05:00,11.4,74,2.8,100,0.0,8.4,74.0,3.8,100.0,0.0
7,2024-01-01 02:00:00-05:00,9.7,74,2.9,100,0.0,6.3,76.0,3.8,100.0,0.0
8,2024-01-01 03:00:00-05:00,8.1,76,2.8,65,0.0,8.9,74.0,3.1,100.0,0.0
9,2024-01-01 04:00:00-05:00,7.5,88,0.7,91,0.0,11.4,74.0,2.6,100.0,0.0


## 5. Rolling statistics

A 24-hour rolling mean and standard deviation for PM2.5 smooth out hourly
noise and capture the recent trend level and volatility — both useful
signals beyond single-point lags.

We use a **trailing** window here (not centered, unlike the exploratory plot
in Notebook 3) since at prediction time we only ever have access to past
values, never future ones.

In [6]:
df['PM2.5_rolling_mean_24h'] = df['PM2.5'].rolling(window=24).mean()
df['PM2.5_rolling_std_24h'] = df['PM2.5'].rolling(window=24).std()

df[['time', 'PM2.5', 'PM2.5_rolling_mean_24h', 'PM2.5_rolling_std_24h']].head(26)

,time,PM2.5,PM2.5_rolling_mean_24h,PM2.5_rolling_std_24h
0,2023-12-31 19:00:00-05:00,19.3,NaN,NaN
1,2023-12-31 20:00:00-05:00,21.7,NaN,NaN
2,2023-12-31 21:00:00-05:00,24.6,NaN,NaN
3,2023-12-31 22:00:00-05:00,27.8,NaN,NaN
4,2023-12-31 23:00:00-05:00,30.5,NaN,NaN
5,2024-01-01 00:00:00-05:00,31.9,NaN,NaN
6,2024-01-01 01:00:00-05:00,28.3,NaN,NaN
7,2024-01-01 02:00:00-05:00,22.8,NaN,NaN
8,2024-01-01 03:00:00-05:00,20.4,NaN,NaN
9,2024-01-01 04:00:00-05:00,19.4,NaN,NaN


## 6. Calendar features

`Hour`, `Month`, and `Weekend` are already numeric/boolean and usable as-is.
`Weekday` and `Season` are categorical text — we one-hot encode these into
separate 0/1 columns so models can use them.

In [7]:
df = pd.get_dummies(df, columns=['Weekday', 'Season'], prefix=['Weekday', 'Season'])

# Weekend was boolean — convert to 0/1 int for consistency with the model input
df['Weekend'] = df['Weekend'].astype(int)

calendar_cols = ['Hour', 'Month', 'Weekend', 'ISO Week'] + \
    [c for c in df.columns if c.startswith('Weekday_') or c.startswith('Season_')]

print("Calendar feature columns:")
print(calendar_cols)

Calendar feature columns:
['Hour', 'Month', 'Weekend', 'ISO Week', 'Weekday_Friday', 'Weekday_Monday', 'Weekday_Saturday', 'Weekday_Sunday', 'Weekday_Thursday', 'Weekday_Tuesday', 'Weekday_Wednesday', 'Season_Fall', 'Season_Spring', 'Season_Summer', 'Season_Winter']


## 7. Drop rows with NaN from lagging/rolling

The first 24 rows now have missing values in the lag/rolling columns (there's
no "24 hours ago" for the very first row in the dataset). We drop these —
losing 24 rows out of 17,568 is negligible.

In [8]:
before = df.shape[0]
df = df.dropna().reset_index(drop=True)
after = df.shape[0]

print(f"Rows before dropna: {before}")
print(f"Rows after dropna:  {after}")
print(f"Rows dropped:       {before - after}")

Rows before dropna: 17568
Rows after dropna:  17544
Rows dropped:       24


## 8. Assemble the final feature set

We keep: PM2.5 (target), all engineered lag features, rolling stats, and
calendar features. Raw Wind Direction and raw Precipitation are excluded as
standalone features per Notebook 4's recommendation (their lagged version of
Precipitation is kept since it passed Granger causality; Wind Direction is
dropped entirely as it did not).

In [9]:
feature_columns = (
    [f'PM2.5(t-{lag})' for lag in pm25_lags]
    + lag_cols
    + ['PM2.5_rolling_mean_24h', 'PM2.5_rolling_std_24h']
    + calendar_cols
)

target_column = 'PM2.5'

print(f"Total features: {len(feature_columns)}")
print(feature_columns)

Total features: 27
['PM2.5(t-1)', 'PM2.5(t-3)', 'PM2.5(t-6)', 'PM2.5(t-12)', 'PM2.5(t-24)', 'Wind Speed(t-3)', 'Relative Humidity(t-2)', 'Temperature(t-6)', 'Cloud Cover(t-2)', 'Precipitation(t-1)', 'PM2.5_rolling_mean_24h', 'PM2.5_rolling_std_24h', 'Hour', 'Month', 'Weekend', 'ISO Week', 'Weekday_Friday', 'Weekday_Monday', 'Weekday_Saturday', 'Weekday_Sunday', 'Weekday_Thursday', 'Weekday_Tuesday', 'Weekday_Wednesday', 'Season_Fall', 'Season_Spring', 'Season_Summer', 'Season_Winter']


In [10]:
model_df = df[['time', target_column] + feature_columns].copy()
model_df.head()

,time,PM2.5,PM2.5(t-1),PM2.5(t-3),PM2.5(t-6),PM2.5(t-12),PM2.5(t-24),Wind Speed(t-3),Relative Humidity(t-2),Temperature(t-6),Cloud Cover(t-2),Precipitation(t-1),PM2.5_rolling_mean_24h,PM2.5_rolling_std_24h,Hour,Month,Weekend,ISO Week,Weekday_Friday,Weekday_Monday,Weekday_Saturday,Weekday_Sunday,Weekday_Thursday,Weekday_Tuesday,Weekday_Wednesday,Season_Fall,Season_Spring,Season_Summer,Season_Winter
0,2024-01-01 19:00:00-05:00,20.4,28.8,10.6,5.7,19.9,19.3,8.0,72.0,7.7,4.0,0.0,19.220833,8.306937,19,1,0,1,False,True,False,False,False,False,False,False,False,False,True
1,2024-01-01 20:00:00-05:00,18.3,20.4,20.4,6.1,27.0,21.7,6.8,79.0,7.5,91.0,0.0,19.079167,8.291797,20,1,0,1,False,True,False,False,False,False,False,False,False,False,True
2,2024-01-01 21:00:00-05:00,14.9,18.3,28.8,6.7,22.5,24.6,7.1,87.0,7.4,32.0,0.0,18.675000,8.247279,21,1,0,1,False,True,False,False,False,False,False,False,False,False,True
3,2024-01-01 22:00:00-05:00,12.0,14.9,20.4,10.6,12.5,27.8,7.1,79.0,7.0,16.0,0.0,18.016667,8.116792,22,1,0,1,False,True,False,False,False,False,False,False,False,False,True
4,2024-01-01 23:00:00-05:00,9.9,12.0,18.3,20.4,8.1,30.5,11.2,70.0,4.9,1.0,0.0,17.158333,7.823205,23,1,0,1,False,True,False,False,False,False,False,False,False,False,True


## 9. Normalize with MinMaxScaler

We scale all numeric feature columns (and the target) to a 0–1 range. Note:
this scaler is fit on the **entire** dataset here for simplicity — in
Notebook 6, when we do a proper train/test split for model evaluation, be
aware that fitting the scaler on all data (including what will become the
test set) is a mild form of data leakage. For rigorous evaluation, refit the
scaler on the training split only; the version saved here is intended for
the deployed dashboard, where there's no held-out test set to worry about.

In [11]:
scaler_columns = feature_columns + [target_column]

scaler = MinMaxScaler()
scaled_values = scaler.fit_transform(model_df[scaler_columns])

scaled_df = pd.DataFrame(scaled_values, columns=scaler_columns)
scaled_df['time'] = model_df['time'].values

scaled_df.head()

,PM2.5(t-1),PM2.5(t-3),PM2.5(t-6),PM2.5(t-12),PM2.5(t-24),Wind Speed(t-3),Relative Humidity(t-2),Temperature(t-6),Cloud Cover(t-2),Precipitation(t-1),PM2.5_rolling_mean_24h,PM2.5_rolling_std_24h,Hour,Month,Weekend,ISO Week,Weekday_Friday,Weekday_Monday,Weekday_Saturday,Weekday_Sunday,Weekday_Thursday,Weekday_Tuesday,Weekday_Wednesday,Season_Fall,Season_Spring,Season_Summer,Season_Winter,PM2.5,time
0,0.386921,0.138965,0.072207,0.265668,0.257493,0.139616,0.670588,0.427320,0.04,0.0,0.350494,0.343596,0.826087,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.272480,2024-01-02 00:00:00
1,0.272480,0.272480,0.077657,0.362398,0.290191,0.118674,0.752941,0.423818,0.91,0.0,0.347648,0.342939,0.869565,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.243869,2024-01-02 01:00:00
2,0.243869,0.386921,0.085831,0.301090,0.329700,0.123909,0.847059,0.422067,0.32,0.0,0.339528,0.341005,0.913043,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.197548,2024-01-02 02:00:00
3,0.197548,0.272480,0.138965,0.164850,0.373297,0.123909,0.752941,0.415061,0.16,0.0,0.326302,0.335337,0.956522,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.158038,2024-01-02 03:00:00
4,0.158038,0.243869,0.272480,0.104905,0.410082,0.195462,0.647059,0.378284,0.01,0.0,0.309057,0.322584,1.000000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.129428,2024-01-02 04:00:00


## 10. Save artifacts

Save the scaler and the feature column list — both are required later by
Notebook 8 (Deployment Preparation) and the Streamlit dashboard to correctly
transform new incoming data the same way.

In [12]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(scaler, MODELS_DIR / "scaler.pkl")
joblib.dump(feature_columns, MODELS_DIR / "feature_columns.pkl")

print("Saved:")
print("-", MODELS_DIR / "scaler.pkl")
print("-", MODELS_DIR / "feature_columns.pkl")

Saved:
- C:\Users\pc\Desktop\AirSenseAI\models\scaler.pkl
- C:\Users\pc\Desktop\AirSenseAI\models\feature_columns.pkl


## 11. Save the model-ready dataset

We also save both the unscaled and scaled versions of the final feature
table as a checkpoint for Notebook 6 (Model Training) — some models (like
SARIMA and Prophet) work directly on raw values, while LSTM/BiLSTM typically
need the scaled version.

In [13]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

model_df.to_csv(PROCESSED_DIR / "airsense_features.csv", index=False)
scaled_df.to_csv(PROCESSED_DIR / "airsense_features_scaled.csv", index=False)

print("Saved:")
print("-", PROCESSED_DIR / "airsense_features.csv", "| shape:", model_df.shape)
print("-", PROCESSED_DIR / "airsense_features_scaled.csv", "| shape:", scaled_df.shape)

Saved:
- C:\Users\pc\Desktop\AirSenseAI\data\processed\airsense_features.csv | shape: (17544, 29)
- C:\Users\pc\Desktop\AirSenseAI\data\processed\airsense_features_scaled.csv | shape: (17544, 29)


## 12. Summary

At this point we have:

- ✅ Built PM2.5 autoregressive lags (t-1, t-3, t-6, t-12, t-24)
- ✅ Built weather lag features at Notebook 4's data-driven optimal lags
  (not arbitrary guesses)
- ✅ Built 24-hour trailing rolling mean and std for PM2.5
- ✅ Encoded calendar features (Weekday, Season as dummies; Hour, Month,
  Weekend, ISO Week numeric)
- ✅ Dropped the 24 rows with unavoidable NaN from lagging
- ✅ Normalized all features + target with MinMaxScaler
- ✅ Saved `scaler.pkl` and `feature_columns.pkl` to `models/`
- ✅ Saved both raw and scaled feature tables to `data/processed/`

**Next step → Notebook 6 (Model Training):**
Train and evaluate Persistence, SARIMA, Prophet, LSTM, and BiLSTM models on
this feature set, using MAE, RMSE, MAPE, and R² — starting with a proper
time-based train/test split (never shuffle time series data randomly).